In [1]:
import pandas as pd
import numpy as np
import os
import re
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 100)

# Paths
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MAPPINGS_DIR = PROJECT_ROOT / "data" / "mappings"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MAPPINGS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print("Setup complete.")

Project root: C:\Users\vises\DAproj\india-healthcare
Setup complete.


In [2]:
df_raw = pd.read_csv(RAW_DIR / "India_Change.csv", encoding='utf-8-sig')

print("=" * 60)
print("RAW DATA LOADED")
print("=" * 60)
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print(f"\nData types:")
print(df_raw.dtypes)
print(f"\nFirst 3 rows:")
df_raw.head(3)

RAW DATA LOADED
Shape: (66976, 10)
Columns: ['State', 'ST_CEN_CD', 'District Name', 'DISTRICT', 'DT_CEN_CD', 'Category', 'Indicator', 'NFHS 5', 'NFHS 4', 'Change']

Data types:
State                str
ST_CEN_CD          int64
District Name        str
DISTRICT             str
DT_CEN_CD         object
Category             str
Indicator            str
NFHS 5           float64
NFHS 4           float64
Change           float64
dtype: object

First 3 rows:


,State,ST_CEN_CD,District Name,DISTRICT,DT_CEN_CD,Category,Indicator,NFHS 5,NFHS 4,Change
0,Andaman & Nicobar Island,35,Nicobar,Nicobar_1,1.0,Anaemia among Children and Women,All women age 15-19 years who are anaemic (%),48.0,66.2,18.2
1,Andaman & Nicobar Island,35,Nicobar,Nicobar_1,1.0,Anaemia among Children and Women,All women age 15-49 years who are anaemic (%),38.3,55.2,16.9
2,Andaman & Nicobar Island,35,Nicobar,Nicobar_1,1.0,Anaemia among Children and Women,Children age 6-59 months who are anaemic,37.7,50.5,12.8


In [3]:
print("DATA STRUCTURE OVERVIEW")
print(f"\nUnique States/UTs: {df_raw['State'].nunique()}")
print(f"Unique Districts:  {df_raw['District Name'].nunique()}")
print(f"Unique Categories: {df_raw['Category'].nunique()}")
print(f"Unique Indicators: {df_raw['Indicator'].nunique()}")

print(f"\n--- CATEGORIES ({df_raw['Category'].nunique()}) ---")
for i, cat in enumerate(sorted(df_raw['Category'].unique()), 1):
    count = df_raw[df_raw['Category'] == cat]['Indicator'].nunique()
    print(f"  {i:2d}. {cat} ({count} indicators)")

print(f"\n--- INDICATORS PER DISTRICT ---")
indicators_per_district = df_raw.groupby('District Name')['Indicator'].count()
print(f"  Min: {indicators_per_district.min()}")
print(f"  Max: {indicators_per_district.max()}")
print(f"  Median: {indicators_per_district.median()}")
print(f"  Districts with < 100 indicators: {(indicators_per_district < 100).sum()}")

DATA STRUCTURE OVERVIEW

Unique States/UTs: 37
Unique Districts:  639
Unique Categories: 19
Unique Indicators: 105

--- CATEGORIES (19) ---
   1. Anaemia among Children and Women (5 indicators)
   2. Blood Sugar Level among men (4 indicators)
   3. Blood Sugar Level among women (3 indicators)
   4. Characteristics of Women (2 indicators)
   5. Child Feeding Practices and Nutritional Status of Children (11 indicators)
   6. Child Vaccinations and Vitamin A Supplementation (11 indicators)
   7. Current Use of Family Planning Methods (8 indicators)
   8. Delivery Care (7 indicators)
   9. Hypertension among men (3 indicators)
  10. Hypertension among women (3 indicators)
  11. Marriage and Fertility (4 indicators)
  12. Maternal and Child Health (10 indicators)
  13. Nutritional Status of Women (3 indicators)
  14. Population and Household Profile (13 indicators)
  15. Quality of Family Planning Services (2 indicators)
  16. Screening for Cancer among Women (3 indicators)
  17. Tobacco Us

In [4]:
# Working on a copy
df = df_raw.copy()

df.columns = df.columns.str.strip()
df = df.rename(columns={
    'State': 'state',
    'ST_CEN_CD': 'state_census_code',
    'District Name': 'district_name',
    'DISTRICT': 'district_id',
    'DT_CEN_CD': 'district_census_code',
    'Category': 'category',
    'Indicator': 'indicator',
    'NFHS 5': 'nfhs5_value',
    'NFHS 4': 'nfhs4_value',
    'Change': 'change'
})

for col in ['state', 'district_name', 'district_id', 'category', 'indicator']:
    df[col] = df[col].astype(str).str.strip()

print("Columns renamed:")
print(list(df.columns))
print(f"\nShape: {df.shape}")
df.head(3)

Columns renamed:
['state', 'state_census_code', 'district_name', 'district_id', 'district_census_code', 'category', 'indicator', 'nfhs5_value', 'nfhs4_value', 'change']

Shape: (66976, 10)


,state,state_census_code,district_name,district_id,district_census_code,category,indicator,nfhs5_value,nfhs4_value,change
0,Andaman & Nicobar Island,35,Nicobar,Nicobar_1,1.0,Anaemia among Children and Women,All women age 15-19 years who are anaemic (%),48.0,66.2,18.2
1,Andaman & Nicobar Island,35,Nicobar,Nicobar_1,1.0,Anaemia among Children and Women,All women age 15-49 years who are anaemic (%),38.3,55.2,16.9
2,Andaman & Nicobar Island,35,Nicobar,Nicobar_1,1.0,Anaemia among Children and Women,Children age 6-59 months who are anaemic,37.7,50.5,12.8


In [5]:
#Handling Numeric Columns
print("NUMERIC COLUMN ANALYSIS")

for col in ['nfhs5_value', 'nfhs4_value', 'change']:
    print(f"\n--- {col} ---")
    print(f"  Current dtype: {df[col].dtype}")
    
    if df[col].dtype == 'object':
        # Find non-numeric values
        mask = pd.to_numeric(df[col], errors='coerce').isna() & df[col].notna()
        non_numeric = df.loc[mask, col].unique()
        print(f"  Non-numeric values found: {len(non_numeric)}")
        if len(non_numeric) > 0:
            print(f"  Examples: {non_numeric[:15]}")
    else:
        print(f"  Already numeric. NaN count: {df[col].isna().sum()}")
        print(f"  Zero count: {(df[col] == 0).sum()}")

NUMERIC COLUMN ANALYSIS

--- nfhs5_value ---
  Current dtype: float64
  Already numeric. NaN count: 0
  Zero count: 5415

--- nfhs4_value ---
  Current dtype: float64
  Already numeric. NaN count: 0
  Zero count: 26460

--- change ---
  Current dtype: float64
  Already numeric. NaN count: 0
  Zero count: 6016


In [6]:
for col in ['nfhs5_value', 'nfhs4_value', 'change']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("After conversion:")
print(df[['nfhs5_value', 'nfhs4_value', 'change']].describe().round(2))

print(f"\nNaN counts after conversion:")
print(f"  nfhs5_value: {df['nfhs5_value'].isna().sum()}")
print(f"  nfhs4_value: {df['nfhs4_value'].isna().sum()}")
print(f"  change:      {df['change'].isna().sum()}")

After conversion:
       nfhs5_value  nfhs4_value    change
count     66976.00     66976.00  66976.00
mean         89.27        68.90     10.29
std         435.31       415.29     24.39
min           0.00         0.00    -96.60
25%           5.80         0.00      0.00
50%          28.20         7.00      4.10
75%          73.40        53.80     14.90
max       18578.00     31457.00    778.50

NaN counts after conversion:
  nfhs5_value: 0
  nfhs4_value: 0
  change:      0


In [7]:
#In this dataset, 0 often means "data not available" rather than an actual zero value (e.g., a district can't have 0% anaemia). 
#We need to identify which zeros are real and which represent missing data.

print("=" * 60)
print("ZERO VALUE ANALYSIS")
print("=" * 60)

# Check indicators where 0 is suspicious
zero_check = df[df['nfhs5_value'] == 0].groupby('indicator').size().sort_values(ascending=False)
print(f"\nIndicators with NFHS-5 value = 0 (top 20):")
print(f"{'Indicator':<75} {'Count':>5}")
print("-" * 82)
for indicator, count in zero_check.head(20).items():
    print(f"{indicator[:75]:<75} {count:>5}")

# Count how many districts have ALL zeros for both NFHS 4 and NFHS 5
both_zero = df[(df['nfhs5_value'] == 0) & (df['nfhs4_value'] == 0)]
print(f"\nRows where BOTH NFHS-5 and NFHS-4 are 0: {len(both_zero)}")
print("(These are likely 'data not available' rather than real zeros)")

ZERO VALUE ANALYSIS

Indicators with NFHS-5 value = 0 (top 20):
Indicator                                                                   Count
----------------------------------------------------------------------------------
Nonbreastfeeding children age 6-23 months receiving an adequate diet (%)      589
Children age 6-8 months receiving solid or semisolid food and breastmilk (%   583
Children born at home who were taken to a health facility for a checkup wit   461
Children with diarrhoea in the 2 weeks preceding the survey who received zi   441
Children with diarrhoea in the 2 weeks preceding the survey taken to a heal   441
Children with diarrhoea in the 2 weeks preceding the survey who received or   441
Male sterilization (%)                                                        303
Children age 12-23 months who received most of their vaccinations in a priv   254
Children under age 6 months exclusively breastfed (%)                         236
Children with fever or symptoms o

In [8]:
# If BOTH nfhs5 and nfhs4 are 0, it's almost certainly "not available"
# For percentage indicators, a true 0% across a district is extremely rare
# We'll replace these with NaN

# Replace, where both NFHS-5 and NFHS-4 are 0, set all three to NaN
mask_both_zero = (df['nfhs5_value'] == 0) & (df['nfhs4_value'] == 0)
df.loc[mask_both_zero, ['nfhs5_value', 'nfhs4_value', 'change']] = np.nan

print(f"Rows converted from (0, 0) to NaN: {mask_both_zero.sum()}")
print(f"\nRemaining zeros in nfhs5_value: {(df['nfhs5_value'] == 0).sum()}")
print("(These are either real zeros or single-side missing , we keep them)")

Rows converted from (0, 0) to NaN: 5393

Remaining zeros in nfhs5_value: 22
(These are either real zeros or single-side missing , we keep them)


In [9]:
indicators = df['indicator'].unique()
print(f"Total unique indicators: {len(indicators)}")

# This function creates a clean, short column name from the long indicator text
def create_short_name(indicator_text):
    """Convert long indicator text to a short, clean column name."""
    text = indicator_text.lower().strip()
    if 'antenatal' in text and 'first trimester' in text:
        return 'anc_first_trimester'
    if 'at least 4 antenatal' in text:
        return 'anc_4plus_visits'
    if 'iron folic acid' in text and '180' in text:
        return 'ifa_180_days'
    if 'iron folic acid' in text and '100' in text:
        return 'ifa_100_days'
    if 'institutional births' in text and 'public' not in text:
        return 'institutional_births'
    if 'institutional births in public' in text:
        return 'births_public_facility'
    if 'births attended by skilled' in text:
        return 'skilled_birth_attendance'
    if 'caesarean section' in text and 'public' in text:
        return 'csection_public'
    if 'caesarean section' in text and 'private' in text:
        return 'csection_private'
    if 'delivered by caesarean section' in text and 'public' not in text and 'private' not in text:
        return 'csection_rate'
    if 'home births' in text and 'skilled' in text:
        return 'home_births_skilled'
    if 'out-of-pocket' in text or 'out of pocket' in text:
        return 'oop_delivery_cost_rs'
    if 'postnatal care' in text and 'mother' in text:
        return 'postnatal_care_mother'
    if 'postnatal care' in text and 'children' in text.lower():
        return 'postnatal_care_child'
    if 'mothers who received postnatal' in text:
        return 'postnatal_care_mother'
    if 'children who received postnatal' in text or 'children born at home' in text.lower():
        if 'home' in text:
            return 'child_home_birth_checkup'
        return 'postnatal_care_child'
    if 'neonatal tetanus' in text:
        return 'tetanus_protection'
    if 'mother and child protection' in text or 'mcp card' in text.lower():
        return 'mcp_card'
    

    if 'stunted' in text:
        return 'child_stunting'
    if 'severely wasted' in text:
        return 'child_severe_wasting'
    if 'wasted' in text and 'severely' not in text and 'under 5' in text:
        return 'child_wasting'
    if 'underweight' in text:
        return 'child_underweight'
    if 'overweight' in text and 'children' in text.lower():
        return 'child_overweight'
    if 'exclusively breastfed' in text:
        return 'exclusive_breastfeeding'
    if 'breastfed within one hour' in text:
        return 'early_breastfeeding'
    if 'breastfeeding children' in text and 'adequate diet' in text:
        return 'bf_children_adequate_diet'
    if 'nonbreastfeeding' in text and 'adequate diet' in text:
        return 'non_bf_children_adequate_diet'
    if 'total children' in text and 'adequate diet' in text:
        return 'children_adequate_diet'
    if 'solid or semisolid' in text:
        return 'complementary_feeding'
    

    if 'fully vaccinated' in text and 'card or mother' in text:
        return 'full_vaccination'
    if 'fully vaccinated' in text and 'card only' in text:
        return 'full_vaccination_card_only'
    if 'bcg' in text:
        return 'vaccine_bcg'
    if 'measles' in text and 'first dose' in text:
        return 'vaccine_measles1'
    if 'measles' in text and 'second dose' in text:
        return 'vaccine_measles2'
    if 'polio' in text:
        return 'vaccine_polio3'
    if 'penta or dpt' in text or 'penta or DPT' in text.lower():
        return 'vaccine_dpt3'
    if 'hepatitis' in text:
        return 'vaccine_hepatitis_b3'
    if 'rotavirus' in text:
        return 'vaccine_rotavirus'
    if 'vaccinations in a public' in text:
        return 'vaccination_public_facility'
    if 'vitamin a' in text:
        return 'vitamin_a_dose'
    

    if 'children age 6-59' in text and 'anaemic' in text:
        return 'anaemia_children'
    if 'all women age 15-49' in text and 'anaemic' in text:
        return 'anaemia_all_women'
    if 'all women age 15-19' in text and 'anaemic' in text:
        return 'anaemia_women_15_19'
    if 'nonpregnant' in text and 'anaemic' in text:
        return 'anaemia_nonpregnant_women'
    if 'pregnant women' in text and 'anaemic' in text:
        return 'anaemia_pregnant_women'
    

    if 'male' in text.lower() and 'blood sugar' in text and 'high or very high' in text:
        return 'men_high_blood_sugar'
    if 'male' in text.lower() and 'blood sugar' in text and 'very high' in text and 'high or' not in text:
        return 'men_very_high_blood_sugar'
    if 'male' in text.lower() and 'blood sugar' in text and '141-160' in text:
        return 'men_blood_sugar_141_160'
    if 'female' in text and 'blood sugar' in text and 'high or very high' in text:
        return 'women_high_blood_sugar'
    if 'female' in text and 'blood sugar' in text and 'very high' in text and 'high or' not in text:
        return 'women_very_high_blood_sugar'
    if 'female' in text and 'blood sugar' in text and '141-160' in text:
        return 'women_blood_sugar_141_160'
    

    if 'male' in text.lower() and 'elevated blood pressure or taking' in text:
        return 'men_hypertension'
    if 'male' in text.lower() and 'mildly elevated' in text:
        return 'men_mild_hypertension'
    if 'male' in text.lower() and 'moderately or severely elevated' in text:
        return 'men_severe_hypertension'
    if 'female' in text and 'elevated blood pressure or taking' in text:
        return 'women_hypertension'
    if 'female' in text and 'mildly elevated' in text:
        return 'women_mild_hypertension'
    if 'female' in text and 'moderately or severely elevated' in text:
        return 'women_severe_hypertension'
    

    if 'any method' in text and 'family planning' not in text:
        return 'fp_any_method'
    if 'any modern method' in text:
        return 'fp_modern_method'
    if 'female sterilization' in text:
        return 'fp_female_sterilization'
    if 'male sterilization' in text:
        return 'fp_male_sterilization'
    if 'iud' in text.lower() or 'ppiud' in text.lower():
        return 'fp_iud'
    if text.startswith('pill') or (text.startswith('pill') is False and 'pill (%)' in text):
        return 'fp_pill'
    if 'condom' in text:
        return 'fp_condom'
    if 'injectable' in text:
        return 'fp_injectable'
    if 'unmet need' in text and 'spacing' in text:
        return 'unmet_need_spacing'
    if 'unmet need' in text and 'total' in text.lower():
        return 'unmet_need_total'
    if 'health worker' in text and 'family planning' in text:
        return 'fp_health_worker_contact'
    if 'current users' in text and 'side effects' in text:
        return 'fp_told_side_effects'
    

    if 'married before age 18' in text:
        return 'women_married_before_18'
    if 'men age 25-29' in text and 'married' in text:
        return 'men_married_before_21'
    if 'already mothers or pregnant' in text:
        return 'teen_pregnancy'
    if 'third or higher order' in text:
        return 'births_third_or_higher'
    if 'sex ratio at birth' in text:
        return 'sex_ratio_at_birth'
    if 'total fertility rate' in text:
        return 'total_fertility_rate'
    if 'hygienic methods' in text and 'menstrual' in text:
        return 'menstrual_hygiene'
    

    if 'body mass index' in text and 'below normal' in text:
        return 'women_underweight_bmi'
    if 'overweight or obese' in text and ('women' in text.lower() or 'female' in text.lower()):
        return 'women_overweight_obese'
    if 'waist' in text and 'hip' in text:
        return 'women_high_waist_hip_ratio'
    if 'men' in text.lower() and 'body mass' in text and 'below' in text:
        return 'men_underweight_bmi'
    if 'men' in text.lower() and 'overweight or obese' in text:
        return 'men_overweight_obese'
    

    if 'clean fuel' in text:
        return 'hh_clean_fuel'
    if 'iodized salt' in text:
        return 'hh_iodized_salt'
    if 'improved sanitation' in text:
        return 'hh_improved_sanitation'
    if 'improved drinking' in text:
        return 'hh_improved_water'
    if 'electricity' in text:
        return 'hh_electricity'
    if 'health insurance' in text:
        return 'hh_health_insurance'
    if 'population below age 15' in text:
        return 'pop_below_15'
    if 'population age 6' in text and 'above' in text:
        return 'pop_above_6'
    if 'birth was registered' in text:
        return 'birth_registration'
    if 'death' in text and 'registered' in text:
        return 'death_registration'
    if 'preprimary' in text or 'pre-primary' in text:
        return 'preschool_attendance'
    if 'female population' in text and 'ever attended school' in text:
        return 'female_school_attendance'
    if 'sex ratio' in text and 'birth' not in text and 'household' in text.lower():
        return 'household_sex_ratio'
    

    if '10 or more years of schooling' in text:
        return 'women_10yr_schooling'
    if 'women who are literate' in text:
        return 'women_literacy'
    if 'women who have' in text and 'mobile' in text:
        return 'women_mobile_phone'
    if 'women who have' in text and 'bank' in text or 'savings account' in text:
        return 'women_bank_account'
    

    if 'women' in text.lower() and 'tobacco' in text:
        return 'women_tobacco'
    if 'men' in text.lower() and 'tobacco' in text:
        return 'men_tobacco'
    if 'alcohol' in text:
        return 'men_alcohol'
    

    if 'diarrhoea' in text and 'ors' in text.lower():
        return 'child_diarrhoea_ors'
    if 'diarrhoea' in text and 'zinc' in text:
        return 'child_diarrhoea_zinc'
    if 'diarrhoea' in text and 'ors and zinc' in text.lower():
        return 'child_diarrhoea_ors_zinc'
    if 'acute respiratory' in text or ('ari' in text.lower() and 'facility' in text):
        return 'child_ari_treatment'
    if 'diarrhoea' in text and 'advice or treatment' in text:
        return 'child_diarrhoea_treatment'
    

    # Remove special characters, truncate
    slug = re.sub(r'[^a-z0-9\s]', '', text)
    slug = re.sub(r'\s+', '_', slug.strip())
    return slug[:60]  # Truncate to 60 chars



indicator_list = sorted(df['indicator'].unique())
indicator_mapping = {}
for ind in indicator_list:
    short = create_short_name(ind)
    indicator_mapping[ind] = short

# Check for duplicates in short names
short_names = list(indicator_mapping.values())
duplicates = [name for name in set(short_names) if short_names.count(name) > 1]

if duplicates:
    print(f"DUPLICATE SHORT NAMES FOUND: {duplicates}")
    for dup in duplicates:
        originals = [k for k, v in indicator_mapping.items() if v == dup]
        for orig in originals:
            print(f"   '{orig}' -> '{dup}'")
else:
    print("No duplicate short names, all mappings are unique!")

print(f"\nTotal indicators mapped: {len(indicator_mapping)}")
print(f"\n--- FIRST 20 MAPPINGS ---")
for i, (long_name, short_name) in enumerate(list(indicator_mapping.items())[:20], 1):
    print(f"  {i:2d}. {short_name:<40} ← {long_name[:70]}")

Total unique indicators: 105
DUPLICATE SHORT NAMES FOUND: ['men_hypertension', 'men_high_blood_sugar', 'child_ari_treatment', 'men_mild_hypertension', 'men_blood_sugar_141_160', 'men_very_high_blood_sugar', 'bf_children_adequate_diet', 'men_severe_hypertension', 'men_alcohol']
   'Female Elevated blood pressure or taking medicine to control blood pressure (%)' -> 'men_hypertension'
   'Male Elevated blood pressure or taking medicine to control blood pressure (%)' -> 'men_hypertension'
   'Female Blood sugar level  high or very high (>140 mg/dl) or taking medicine to control blood sugar level (%)' -> 'men_high_blood_sugar'
   'Male Blood sugar level  high or very high (>140 mg/dl) or taking medicine to control blood sugar level (%)' -> 'men_high_blood_sugar'
   'Children with fever or symptoms of ARI in the 2 weeks preceding the survey taken to a health facility or health provider (%)' -> 'child_ari_treatment'
   'Prevalence of symptoms of acute respiratory infection (ARI) in the 2 week

In [10]:
#Fixing duplicates

# 1. Female hypertension matched "male" because "fe-MALE" contains "male"
indicator_mapping["Female Mildly elevated blood pressure (Systolic 140-159 mm of Hg and/or Diastolic 90-99 mm of Hg) (%)"] = "women_mild_hypertension"
indicator_mapping["Female Elevated blood pressure or taking medicine to control blood pressure (%)"] = "women_hypertension"
indicator_mapping["Female Moderately or severely elevated blood pressure (%)"] = "women_severe_hypertension"

# 2. Female blood sugar matched "male" for same reason
indicator_mapping["Female Blood sugar level  high (141-160 mg/dl) (%)"] = "women_blood_sugar_141_160"
indicator_mapping["Female Blood sugar level  high or very high (>140 mg/dl) or taking medicine to control blood sugar level (%)"] = "women_high_blood_sugar"
indicator_mapping["Female Blood sugar level  very high (>160 mg/dl) (%)"] = "women_very_high_blood_sugar"

# 3. "MaleBlood" (no space) is a duplicate of "Male Blood" — same indicator, typo in data
indicator_mapping["MaleBlood sugar level  high (141-160 mg/dl) (%)"] = "men_blood_sugar_141_160_dup"

# 4. ARI — two different indicators (prevalence vs treatment-seeking)
indicator_mapping["Prevalence of symptoms of acute respiratory infection (ARI) in the 2 weeks preceding the survey (%)"] = "child_ari_prevalence"
indicator_mapping["Children with fever or symptoms of ARI in the 2 weeks preceding the survey taken to a health facility or health provider (%)"] = "child_ari_treatment"

# 5. Women alcohol was matching "men_alcohol" because of the fallback logic
indicator_mapping["Women age 15 years and above who consume alcohol (%)"] = "women_alcohol"
indicator_mapping["Men age 15 years and above who consume alcohol (%)"] = "men_alcohol"

# 6. Breastfeeding vs non-breastfeeding adequate diet
indicator_mapping["Breastfeeding children age 6-23 months receiving an adequate diet (%)"] = "bf_children_adequate_diet"
indicator_mapping["Nonbreastfeeding children age 6-23 months receiving an adequate diet (%)"] = "non_bf_children_adequate_diet"

# Re-check for duplicates
short_names = list(indicator_mapping.values())
duplicates = [name for name in set(short_names) if short_names.count(name) > 1]

if duplicates:
    print(f"Still have duplicates: {duplicates}")
    for dup in duplicates:
        originals = [k for k, v in indicator_mapping.items() if v == dup]
        for orig in originals:
            print(f"   '{orig}' -> '{dup}'")
else:
    print("All short names are unique. Ready to proceed.")

All short names are unique. Ready to proceed.


In [11]:
# Determine which direction is "good" for each indicator
# True = higher value is better (e.g., vaccination rate)
# False = lower value is better (e.g., stunting rate)

lower_is_better = [
    'child_stunting', 'child_wasting', 'child_severe_wasting', 'child_underweight',
    'anaemia_children', 'anaemia_all_women', 'anaemia_women_15_19',
    'anaemia_nonpregnant_women', 'anaemia_pregnant_women',
    'women_underweight_bmi', 'women_high_waist_hip_ratio',
    'men_high_blood_sugar', 'men_very_high_blood_sugar', 'men_blood_sugar_141_160',
    'women_high_blood_sugar', 'women_very_high_blood_sugar', 'women_blood_sugar_141_160',
    'men_hypertension', 'men_mild_hypertension', 'men_severe_hypertension',
    'women_hypertension', 'women_mild_hypertension', 'women_severe_hypertension',
    'women_married_before_18', 'men_married_before_21', 'teen_pregnancy',
    'births_third_or_higher', 'unmet_need_total', 'unmet_need_spacing',
    'women_tobacco', 'men_tobacco', 'men_alcohol',
    'oop_delivery_cost_rs', 'child_overweight', 'men_overweight_obese',
    'csection_rate',  # Over-medicalization concern
]

# Build metadata DataFrame
metadata_rows = []
for long_name, short_name in indicator_mapping.items():
    # Find category from original data
    cat = df[df['indicator'] == long_name]['category'].iloc[0] if len(df[df['indicator'] == long_name]) > 0 else 'Unknown'
    
    # Determine unit
    if 'rs' in short_name or 'cost' in short_name:
        unit = 'INR'
    elif 'rate' in long_name.lower() and 'per 1,000' in long_name.lower():
        unit = 'per_1000'
    elif 'ratio' in long_name.lower():
        unit = 'ratio'
    elif 'total fertility' in long_name.lower():
        unit = 'children_per_woman'
    else:
        unit = 'percentage'
    
    higher_is_good = short_name not in lower_is_better
    
    metadata_rows.append({
        'short_name': short_name,
        'long_name': long_name,
        'category': cat,
        'unit': unit,
        'higher_is_good': higher_is_good
    })

df_indicator_metadata = pd.DataFrame(metadata_rows)
df_indicator_metadata = df_indicator_metadata.sort_values('category').reset_index(drop=True)

# Save
df_indicator_metadata.to_csv(MAPPINGS_DIR / "indicator_metadata.csv", index=False)

print(f"Indicator metadata saved: {len(df_indicator_metadata)} indicators")
print(f"\nCategory distribution:")
print(df_indicator_metadata['category'].value_counts().to_string())
df_indicator_metadata.head(10)

Indicator metadata saved: 105 indicators

Category distribution:
category
Population and Household Profile                              13
Child Feeding Practices and Nutritional Status of Children    11
Child Vaccinations and Vitamin A Supplementation              11
Maternal and Child Health                                     10
Current Use of Family Planning Methods                         8
Delivery Care                                                  7
Treatment of Childhood Diseases                                7
Anaemia among Children and Women                               5
Blood Sugar Level among men                                    4
Marriage and Fertility                                         4
Tobacco Use and Alcohol Consumption among Adults               4
Blood Sugar Level among women                                  3
Hypertension among men                                         3
Hypertension among women                                       3
Nutritional Stat

,short_name,long_name,category,unit,higher_is_good
0,anaemia_women_15_19,All women age 15-19 years who are anaemic (%),Anaemia among Children and Women,percentage,False
1,anaemia_all_women,All women age 15-49 years who are anaemic (%),Anaemia among Children and Women,percentage,False
2,anaemia_pregnant_women,Pregnant women age 15-49 years who are anaemic,Anaemia among Children and Women,percentage,False
3,anaemia_nonpregnant_women,Nonpregnant women age 15-49 years who are anaemic,Anaemia among Children and Women,percentage,False
4,anaemia_children,Children age 6-59 months who are anaemic,Anaemia among Children and Women,percentage,False
5,men_blood_sugar_141_160_dup,MaleBlood sugar level high (141-160 mg/dl) (%),Blood Sugar Level among men,percentage,True
6,men_very_high_blood_sugar,Male Blood sugar level very high (>160 mg/dl) (%),Blood Sugar Level among men,percentage,False
7,men_high_blood_sugar,Male Blood sugar level high or very high (>140 mg/dl) or taking medicine to...,Blood Sugar Level among men,percentage,False
8,men_blood_sugar_141_160,Male Blood sugar level high (141-160 mg/dl) (%),Blood Sugar Level among men,percentage,False
9,women_very_high_blood_sugar,Female Blood sugar level very high (>160 mg/dl) (%),Blood Sugar Level among women,percentage,False


In [12]:
# Define Indian geographic regions
region_mapping = {
    # North
    'Jammu & Kashmir': 'North', 'Himachal Pradesh': 'North', 'Punjab': 'North',
    'Haryana': 'North', 'NCT of Delhi': 'North', 'Uttarakhand': 'North',
    'Uttar Pradesh': 'North', 'Rajasthan': 'North', 'Chandigarh': 'North',
    'Ladakh': 'North',
    
    # South
    'Andhra Pradesh': 'South', 'Karnataka': 'South', 'Kerala': 'South',
    'Tamil Nadu': 'South', 'Telangana': 'South', 'Puducherry': 'South',
    'Lakshadweep': 'South',
    
    # East
    'Bihar': 'East', 'Jharkhand': 'East', 'Odisha': 'East',
    'West Bengal': 'East',
    
    # West
    'Gujarat': 'West', 'Maharashtra': 'West', 'Goa': 'West',
    'Dadra and Nagar Haveli and Daman and Diu': 'West',
    'Dadra & Nagar Haveli and Daman & Diu': 'West',
    'Dadra & Nagar Haveli': 'West',
    'Daman & Diu': 'West',
    
    # Central
    'Madhya Pradesh': 'Central', 'Chhattisgarh': 'Central',
    
    # Northeast
    'Assam': 'Northeast', 'Arunachal Pradesh': 'Northeast',
    'Manipur': 'Northeast', 'Meghalaya': 'Northeast',
    'Mizoram': 'Northeast', 'Nagaland': 'Northeast',
    'Sikkim': 'Northeast', 'Tripura': 'Northeast',
    
    # Islands
    'Andaman & Nicobar Island': 'Islands',
    'Andaman & Nicobar Islands': 'Islands',
}


data_states = df['state'].unique()
unmapped = [s for s in data_states if s not in region_mapping]

if unmapped:
    print(f"States NOT in region mapping: {unmapped}")
else:
    print(f" All {len(data_states)} states/UTs mapped to regions.")

# Add region to dataframe
df['region'] = df['state'].map(region_mapping)

print(f"\nRegion distribution (by district count):")
region_dist = df.groupby('region')['district_name'].nunique()
for region, count in region_dist.sort_values(ascending=False).items():
    print(f"  {region:<12}: {count} districts")

 All 37 states/UTs mapped to regions.

Region distribution (by district count):
  North       : 201 districts
  East        : 111 districts
  South       : 105 districts
  Northeast   : 88 districts
  Central     : 68 districts
  West        : 66 districts
  Islands     : 3 districts


In [13]:
# Map the short names onto the dataframe
df['indicator_short'] = df['indicator'].map(indicator_mapping)

# Check for any unmapped indicators
unmapped_indicators = df[df['indicator_short'].isna()]['indicator'].unique()
if len(unmapped_indicators) > 0:
    print(f"{len(unmapped_indicators)} indicators have no short name mapping:")
    for ind in unmapped_indicators:
        print(f"   - {ind}")
else:
    print("All indicators mapped to short names.")

print(f"\nDataframe shape: {df.shape}")
df[['state', 'district_name', 'indicator', 'indicator_short', 'nfhs5_value']].head()

All indicators mapped to short names.

Dataframe shape: (66976, 12)


,state,district_name,indicator,indicator_short,nfhs5_value
0,Andaman & Nicobar Island,Nicobar,All women age 15-19 years who are anaemic (%),anaemia_women_15_19,48.0
1,Andaman & Nicobar Island,Nicobar,All women age 15-49 years who are anaemic (%),anaemia_all_women,38.3
2,Andaman & Nicobar Island,Nicobar,Children age 6-59 months who are anaemic,anaemia_children,37.7
3,Andaman & Nicobar Island,Nicobar,Nonpregnant women age 15-49 years who are anaemic,anaemia_nonpregnant_women,38.4
4,Andaman & Nicobar Island,Nicobar,Pregnant women age 15-49 years who are anaemic,anaemia_pregnant_women,NaN


In [14]:
print("PIVOTING TO WIDE FORMAT")
# Each row = one district, each column = one indicator
df_wide = df.pivot_table(
    index=['state', 'state_census_code', 'district_name', 'district_id', 
           'district_census_code', 'region'],
    columns='indicator_short',
    values='nfhs5_value',
    aggfunc='first'  # In case of duplicates, take the first value
).reset_index()

# Flatten the MultiIndex columns
df_wide.columns.name = None

print(f"Wide format created!")
print(f"   Shape: {df_wide.shape}")
print(f"   Rows (districts): {len(df_wide)}")
print(f"   Columns: {len(df_wide.columns)} (metadata + indicators)")

# Preview
print(f"\n   Metadata columns: {list(df_wide.columns[:6])}")
print(f"   First 5 indicator columns: {list(df_wide.columns[6:11])}")
df_wide.head(3)

PIVOTING TO WIDE FORMAT
Wide format created!
   Shape: (649, 110)
   Rows (districts): 649
   Columns: 110 (metadata + indicators)

   Metadata columns: ['state', 'state_census_code', 'district_name', 'district_id', 'district_census_code', 'region']
   First 5 indicator columns: ['anaemia_all_women', 'anaemia_children', 'anaemia_nonpregnant_women', 'anaemia_pregnant_women', 'anaemia_women_15_19']


,state,state_census_code,district_name,district_id,district_census_code,region,anaemia_all_women,anaemia_children,anaemia_nonpregnant_women,anaemia_pregnant_women,anaemia_women_15_19,anc_4plus_visits,anc_first_trimester,bf_children_adequate_diet,birth_registration,births_public_facility,births_third_or_higher,child_ari_prevalence,child_ari_treatment,child_diarrhoea_ors,child_diarrhoea_zinc,child_home_birth_checkup,child_overweight,child_severe_wasting,child_stunting,child_underweight,child_wasting,children_adequate_diet,children_age_1223_months_who_received_most_of_their_vaccinat,children_with_diarrhoea_in_the_2_weeks_preceding_the_survey_,complementary_feeding,csection_private,csection_public,csection_rate,death_registration,early_breastfeeding,ever_undergone_a_breast_examination_for_breast_cancer,ever_undergone_a_screening_test_for_cervical_cancer,ever_undergone_an_oral_cavity_examination_for_oral_cancer,exclusive_breastfeeding,fp_any_method,fp_condom,fp_female_sterilization,fp_health_worker_contact,fp_injectable,fp_iud,fp_male_sterilization,fp_modern_method,fp_pill,fp_told_side_effects,full_vaccination,full_vaccination_card_only,hh_clean_fuel,hh_electricity,hh_health_insurance,hh_improved_sanitation,hh_improved_water,hh_iodized_salt,home_births_skilled,ifa_100_days,ifa_180_days,institutional_births,mcp_card,men_alcohol,men_blood_sugar_141_160_dup,men_high_blood_sugar,men_hypertension,men_mild_hypertension,men_severe_hypertension,men_tobacco,men_very_high_blood_sugar,menstrual_hygiene,non_bf_children_adequate_diet,oop_delivery_cost_rs,pop_above_6,pop_below_15,postnatal_care_child,postnatal_care_mother,preschool_attendance,prevalence_of_diarrhoea_in_the_2_weeks_preceding_the_survey,sex_ratio_at_birth,sex_ratio_of_the_total_population_females_per_1000_males,skilled_birth_attendance,teen_pregnancy,tetanus_protection,unmet_need_spacing,unmet_need_total,vaccination_public_facility,vaccine_bcg,vaccine_dpt3,vaccine_hepatitis_b3,vaccine_measles1,vaccine_measles2,vaccine_polio3,vaccine_rotavirus,vitamin_a_dose,women_10yr_schooling,women_alcohol,women_blood_sugar_141_160,women_high_blood_sugar,women_high_waist_hip_ratio,women_hypertension,women_literacy,women_married_before_18,women_mild_hypertension,women_overweight_obese,women_severe_hypertension,women_tobacco,women_underweight_bmi,women_very_high_blood_sugar
0,Andaman & Nicobar Island,35,Nicobar,Nicobar_1,1.0,Islands,38.3,37.7,38.4,NaN,48.0,71.7,62.8,19.4,98.0,96.7,NaN,1.8,85.7,NaN,NaN,NaN,1.5,7.8,21.6,24.6,15.7,18.7,NaN,NaN,NaN,NaN,10.7,11.5,83.2,55.4,13.2,13.4,5.4,NaN,65.3,4.9,46.4,40.4,1.2,2.7,NaN,57.2,2.0,49.4,64.2,94.1,56.9,97.9,2.7,83.5,98.8,99.4,0.8,72.6,43.9,97.8,97.9,64.5,9.6,15.4,47.0,32.9,11.1,76.8,4.4,100.0,NaN,2278.0,78.0,23.0,92.5,85.1,29.5,5.7,927.0,973.0,98.6,1.8,78.0,3.3,9.5,100.0,80.4,71.9,68.6,67.3,20.7,69.1,3.1,94.9,53.5,29.6,7.4,13.1,62.5,35.4,87.5,11.4,23.2,39.1,8.5,63.5,8.2,3.9
1,Andaman & Nicobar Island,35,North & Middle Andaman,North & Middle Andaman_2,2.0,Islands,62.1,30.4,62.5,NaN,47.8,79.2,74.5,6.5,100.0,95.0,1.5,7.0,NaN,NaN,NaN,NaN,0.8,8.3,27.0,42.8,27.0,5.9,NaN,NaN,NaN,NaN,11.4,12.9,92.6,27.3,0.3,1.7,15.8,NaN,84.1,9.3,48.3,23.2,NaN,6.4,0.6,73.1,7.8,83.2,NaN,NaN,61.3,93.2,2.1,86.4,92.2,99.9,0.7,83.7,24.1,97.7,99.2,45.3,9.1,18.3,32.2,22.6,6.0,70.5,6.9,100.0,NaN,1904.0,82.7,19.8,94.3,92.5,30.1,4.5,844.0,950.0,98.3,3.8,91.1,1.3,5.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,89.6,41.0,5.1,7.2,16.7,79.3,27.4,84.0,15.4,18.4,35.9,4.0,46.8,8.6,6.4
2,Andaman & Nicobar Island,35,South Andaman,South Andaman_3,3.0,Islands,57.7,43.4,57.6,NaN,43.2,85.9,79.4,22.3,96.5,83.8,0.5,NaN,77.3,NaN,NaN,NaN,7.2,3.5,21.1,17.4,12.6,23.5,4.3,NaN,NaN,79.1,29.6,37.1,92.2,51.1,0.7,1.3,8.0,NaN,57.1,10.6,34.0,31.2,0.3,2.8,NaN,50.5,1.8,88.2,76.3,96.6,91.9,99.6,1.2,89.3,97.9,99.7,NaN,81.0,61.9,99.5,98.9,32.8,9.3,18.1,26.9,17.9,6.1,50.8,7.8,98.2,NaN,3460.0,84.7,21.0,89.8,88.1,50.8,6.0,935.0,967.0,96.9,2.8,92.1,8.6,17.6,93.1,100.0,94.8,85.3,81.7,33.7,79.0,NaN,84.0,57.5,1.7,7.5,18.4,78.2,23.0,86.7,17.

In [15]:
# Also creating wide format for NFHS-4 
df_wide_nfhs4 = df.pivot_table(
    index=['state', 'district_name', 'district_id'],
    columns='indicator_short',
    values='nfhs4_value',
    aggfunc='first'
).reset_index()

df_wide_nfhs4.columns.name = None

# Rename indicator columns to have _nfhs4 suffix
meta_cols = ['state', 'district_name', 'district_id']
indicator_cols = [c for c in df_wide_nfhs4.columns if c not in meta_cols]
rename_map = {col: f"{col}_nfhs4" for col in indicator_cols}
df_wide_nfhs4 = df_wide_nfhs4.rename(columns=rename_map)

print(f"NFHS-4 wide format: {df_wide_nfhs4.shape}")

NFHS-4 wide format: (649, 107)


In [16]:
# Extracting unique district info
df_district_meta = df_wide[['state', 'state_census_code', 'district_name', 
                             'district_id', 'district_census_code', 'region']].copy()

df_district_meta = df_district_meta.reset_index(drop=True)
df_district_meta.index.name = 'row_id'

print(f"District metadata: {df_district_meta.shape}")
print(f"\nSample:")
df_district_meta.head()

District metadata: (649, 6)

Sample:


,state,state_census_code,district_name,district_id,district_census_code,region
row_id,,,,,,
0,Andaman & Nicobar Island,35,Nicobar,Nicobar_1,1.0,Islands
1,Andaman & Nicobar Island,35,North & Middle Andaman,North & Middle Andaman_2,2.0,Islands
2,Andaman & Nicobar Island,35,South Andaman,South Andaman_3,3.0,Islands
3,Andhra Pradesh,28,Anantapur,Anantapur_12,12.0,South
4,Andhra Pradesh,28,Chittoor,Chittoor_13,13.0,South


In [17]:
print("DATA QUALITY REPORT")

# Get only indicator columns 
meta_cols = ['state', 'state_census_code', 'district_name', 'district_id', 
             'district_census_code', 'region']
indicator_cols = [c for c in df_wide.columns if c not in meta_cols]

print(f"\nDataset dimensions: {df_wide.shape[0]} districts × {len(indicator_cols)} indicators")

# Missing values per indicator
missing_pct = df_wide[indicator_cols].isnull().mean().sort_values(ascending=False) * 100

print(f"\n MISSING VALUES SUMMARY:")
print(f"   Indicators with 0% missing:   {(missing_pct == 0).sum()}")
print(f"   Indicators with <5% missing:  {(missing_pct < 5).sum()}")
print(f"   Indicators with <20% missing: {(missing_pct < 20).sum()}")
print(f"   Indicators with >50% missing: {(missing_pct > 50).sum()}")

if (missing_pct > 50).sum() > 0:
    print(f"\n HIGH MISSING (>50%):")
    for col, pct in missing_pct[missing_pct > 50].items():
        print(f"   {col:<45} {pct:.1f}%")

print(f"\n TOP 10 INDICATORS BY MISSING %:")
for col, pct in missing_pct.head(10).items():
    print(f"   {col:<45} {pct:.1f}%")

# Districts with most missing values
district_missing = df_wide[indicator_cols].isnull().sum(axis=1)
df_wide_temp = df_wide.copy()
df_wide_temp['missing_count'] = district_missing

print(f"\n DISTRICTS WITH MOST MISSING INDICATORS:")
worst_districts = df_wide_temp.nlargest(10, 'missing_count')[
    ['state', 'district_name', 'missing_count']
]
for _, row in worst_districts.iterrows():
    print(f"   {row['district_name']}, {row['state']}: {row['missing_count']} missing")

df_wide_temp.drop('missing_count', axis=1, inplace=True)

DATA QUALITY REPORT

Dataset dimensions: 649 districts × 104 indicators

 MISSING VALUES SUMMARY:
   Indicators with 0% missing:   0
   Indicators with <5% missing:  83
   Indicators with <20% missing: 90
   Indicators with >50% missing: 6

 HIGH MISSING (>50%):
   non_bf_children_adequate_diet                 91.5%
   complementary_feeding                         90.6%
   child_home_birth_checkup                      71.5%
   child_diarrhoea_ors                           68.4%
   children_with_diarrhoea_in_the_2_weeks_preceding_the_survey_ 68.4%
   child_diarrhoea_zinc                          68.4%

 TOP 10 INDICATORS BY MISSING %:
   non_bf_children_adequate_diet                 91.5%
   complementary_feeding                         90.6%
   child_home_birth_checkup                      71.5%
   child_diarrhoea_ors                           68.4%
   children_with_diarrhoea_in_the_2_weeks_preceding_the_survey_ 68.4%
   child_diarrhoea_zinc                          68.4%
   fp_male_st

In [18]:
print("SAVING PROCESSED FILES")
# 1. Main wide-format dataset (NFHS-5)
path1 = PROCESSED_DIR / "nfhs5_districts_wide.csv"
df_wide.to_csv(path1, index=False)
print(f"{path1.name}: {df_wide.shape}")

# 2. NFHS-4 wide format (for trend comparison)
path2 = PROCESSED_DIR / "nfhs4_districts_wide.csv"
df_wide_nfhs4.to_csv(path2, index=False)
print(f"{path2.name}: {df_wide_nfhs4.shape}")

# 3. Long format cleaned data (useful for SQL loading)
path3 = PROCESSED_DIR / "nfhs_districts_long.csv"
df_long_clean = df[['state', 'state_census_code', 'district_name', 'district_id',
                     'district_census_code', 'region', 'category', 'indicator', 
                     'indicator_short', 'nfhs5_value', 'nfhs4_value', 'change']].copy()
df_long_clean.to_csv(path3, index=False)
print(f"{path3.name}: {df_long_clean.shape}")

# 4. District metadata
path4 = MAPPINGS_DIR / "district_metadata.csv"
df_district_meta.to_csv(path4, index=False)
print(f"{path4.name}: {df_district_meta.shape}")

# 5. Indicator metadata (already saved in Cell 11, confirm)
path5 = MAPPINGS_DIR / "indicator_metadata.csv"
print(f"{path5.name}: {df_indicator_metadata.shape}")

print(f"\nAll files saved to:")
print(f"   {PROCESSED_DIR}")
print(f"   {MAPPINGS_DIR}")

SAVING PROCESSED FILES
nfhs5_districts_wide.csv: (649, 110)
nfhs4_districts_wide.csv: (649, 107)
nfhs_districts_long.csv: (66976, 12)
district_metadata.csv: (649, 6)
indicator_metadata.csv: (105, 5)

All files saved to:
   C:\Users\vises\DAproj\india-healthcare\data\processed
   C:\Users\vises\DAproj\india-healthcare\data\mappings
